In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()
print(f"📍 Python's Current Anchor Point:\n{current_dir}\n")

print("📁 Folders and files Python can currently see:")
print("-" * 50)
for item in current_dir.iterdir():
    # Adding a label so you know if it's looking at a folder or a file
    item_type = "Folder" if item.is_dir() else "File  "
    print(f"[{item_type}] {item.name}")
print("-" * 50)

In [ ]:
import xarray as xr
from pathlib import Path

# Mapping CMIP6 short names to your Random Forest column names
CMIP6_TO_TARGET = {
    "tos": "SST",
    "siconc": "SIA",
    "simass": "SIM",
    "sithick": "SIT",
    "ps": "SAP",
    "psl": "SLP",
    "sos": "SSS"
}

# 1. Point this to the folder containing all those hashed subfolders
FUTURE_DIR = Path.cwd() / "Future_Data" 
OUTPUT_FILE = FUTURE_DIR / "mpi_esm1_2_ssp245_combined.nc"

print("1. Searching for downloaded .nc files...")
# rglob recursively looks inside all those weird hashed subfolders
nc_files = list(FUTURE_DIR.rglob("*.nc"))
print(f"Found {len(nc_files)} files.")

datasets = []

print("\n2. Processing and Renaming Variables...")
for path in nc_files:
    ds = xr.open_dataset(path)
    
    # Your brilliant filter to drop bounds/vertices
    candidates = [
        v for v in ds.data_vars
        if ds[v].ndim == 3 and "bnds" not in ds[v].dims
    ]
    
    if not candidates:
        print(f"Skipping {path.name} (No valid 3D variable found)")
        ds.close()
        continue
        
    main_var = candidates[0]
    
    if main_var not in CMIP6_TO_TARGET:
        print(f"Warning: Unexpected variable '{main_var}' found. Skipping.")
        ds.close()
        continue
        
    target_name = CMIP6_TO_TARGET[main_var]
    print(f"  Mapped '{main_var}' -> '{target_name}' from {path.name[:25]}...")
    
    # Rename the variable to match your Random Forest
    ds = ds[[main_var]].rename({main_var: target_name})
    datasets.append(ds)

print("\n3. Merging Datasets...")
# compat="override" saves us from floating point grid mismatches
combined = xr.merge(datasets, compat="override")

# Rename time to Date to match your pipeline
if "time" in combined.coords:
    combined = combined.rename({"time": "Date"})

for d in datasets:
    d.close()

print("\n4. Saving Final Combined File...")
combined.to_netcdf(OUTPUT_FILE)
print(f"SUCCESS: Saved → {OUTPUT_FILE.name}")

In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

# Physical valid ranges per variable (inclusive)
VALID_RANGES = {
    "SST": (-2.0,   35.0),   # degC  — freezing point of seawater to max tropical SST
    "SIA": (  0.0, 100.0),   # %
    "SIM": (  0.0, 5000.0),  # kg/m²
    "SIT": (  0.0,   20.0),  # m
    "SAP": (50_000, 110_000),# Pa
    "SLP": (87_000, 108_600),# Pa
    "SSS": (  0.0,   45.0),  # PSU
}

# NetCDF standard fill value
NC_FILL_VALUE = 1e20

# Point to your Future_Data folder
FUTURE_DIR = Path.cwd() / "Future_Data"
input_path = FUTURE_DIR / "mpi_esm1_2_ssp245_combined.nc"
output_path = FUTURE_DIR / "mpi_esm1_2_ssp245_cleaned.nc"

if output_path.exists():
    print(f"Skipping — {output_path.name} already exists. Delete it to rerun.")
else:
    print(f"Loading {input_path.name} ...")
    ds = xr.open_dataset(input_path)

    cleaned_vars = {}
    print("\nApplying physical limits and masking fill values:")
    for var in ds.data_vars:
        da = ds[var]

        # 1. Mask fill values
        da = da.where(np.abs(da) < NC_FILL_VALUE * 0.9)

        # 2. Mask values outside physical range
        if var in VALID_RANGES:
            lo, hi = VALID_RANGES[var]
            before = int(da.isnull().sum())
            da = da.where((da >= lo) & (da <= hi))
            out_of_range = int(da.isnull().sum()) - before
            print(f"  {var:5s}: {out_of_range:,} values clipped to NaN (out of range [{lo}, {hi}])")

        cleaned_vars[var] = da

    print("\nSaving cleaned dataset...")
    cleaned = xr.Dataset(cleaned_vars, coords=ds.coords, attrs=ds.attrs)
    ds.close()

    cleaned.to_netcdf(output_path)
    print(f"SUCCESS: Saved → {output_path.name}")

In [ ]:
import numpy as np
import xarray as xr
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay
from pathlib import Path

print("--- INITIALIZING REGRID PIPELINE ---")

OCEAN_VARS = ["SST", "SIA", "SIM", "SIT", "SSS"]  # curved (j, i) grid
ATM_VARS   = ["SAP", "SLP"]                       # regular (lat, lon) grid

# Target: 1° global regular grid — 180 latitude × 360 longitude cells
DS_TARGET = xr.Dataset({
    "lat": (["lat"], np.arange(-89.5, 90.0, 1.0)),   
    "lon": (["lon"], np.arange(0.5, 360.0, 1.0)),    
})

# Path Setup
FUTURE_DIR = Path.cwd() / "Future_Data"
cleaned_path = FUTURE_DIR / "mpi_esm1_2_ssp245_cleaned.nc"
regridded_path = FUTURE_DIR / "mpi_esm1_2_ssp245_regridded.nc"

# ==========================================
# STEP 1: REGRIDDING ONLY
# ==========================================
if regridded_path.exists():
    print(f"\n[1/1] Regridding: SKIPPED (File {regridded_path.name} already exists)")
else:
    print(f"\n[1/1] Regridding: Loading {cleaned_path.name} ...")
    ds = xr.open_dataset(cleaned_path)

    out_lat = DS_TARGET["lat"].values                                       
    out_lon = DS_TARGET["lon"].values                                       
    tgt_lat, tgt_lon = np.meshgrid(out_lat, out_lon, indexing="ij")        
    xi = np.column_stack([tgt_lat.ravel(), tgt_lon.ravel()])               

    regridded = {}

    # --- Ocean vars: curvilinear (j, i) grid ---
    ocean_vars_present = [v for v in OCEAN_VARS if v in ds]
    if ocean_vars_present:
        lat2d = ds["latitude"].values.ravel()
        lon2d = ds["longitude"].values.ravel() % 360   # normalize to 0-360
        valid = np.isfinite(lat2d) & np.isfinite(lon2d)
        src_pts = np.column_stack([lat2d[valid], lon2d[valid]])

        print(f"  Building triangulation for ocean grid ({valid.sum():,} points) ...")
        tri = Delaunay(src_pts)

        for var in ocean_vars_present:
            da = ds[var]                                                    
            ntime = da.shape[0]
            vals = da.values.reshape(ntime, -1)[:, valid].T                
            print(f"    Regridding {var} ({ntime} timesteps) ...")
            out = LinearNDInterpolator(tri, vals)(xi)                      
            result = out.T.reshape(ntime, len(out_lat), len(out_lon))      
            regridded[var] = xr.DataArray(
                result,
                dims=["Date", "lat", "lon"],
                coords={"Date": ds["Date"], "lat": out_lat, "lon": out_lon},
                attrs=da.attrs,
            )

    # --- Atm vars: already on a regular (lat, lon) grid ---
    atm_vars_present = [v for v in ATM_VARS if v in ds]
    if atm_vars_present:
        print(f"  Regridding atmosphere vars ...")
        ds_atm_out = ds[atm_vars_present].interp(lat=out_lat, lon=out_lon, method="linear")
        for var in atm_vars_present:
            regridded[var] = ds_atm_out[var]

    ds_out = xr.Dataset(regridded, attrs=ds.attrs)
    ds.close()

    ds_out.to_netcdf(regridded_path)
    print(f"  SUCCESS: Saved regridded file → {regridded_path.name}")

print("\n--- PIPELINE COMPLETE ---")

In [ ]:
import cdsapi
import zipfile
import tempfile
import os
import shutil
from pathlib import Path

DATASET = "projections-cmip6"

MODELS = [
    "ec_earth3_veg_lr",
    "hadgem3_gc31_ll",
    "mpi_esm1_2_lr",
]

VARIABLES = [
    "sea_surface_temperature",                  # SST (K)
    "sea_ice_area_percentage_on_ocean_grid",    # SIA (%)
    "sea_ice_mass_per_area",                    # SIM (kg/m²)
    "sea_ice_thickness",                        # SIT (m)
    "surface_air_pressure",                     # SAP (Pa)
    "sea_level_pressure",                       # SLP (Pa)
    "sea_surface_salinity",                     # SSS (PSU)
]

YEARS = [str(y) for y in range(1926, 2015)]
MONTHS = [f"{m:02d}" for m in range(1, 13)]

# FIXED for Jupyter compatibility: Uses Current Working Directory
# Adjust the .parent chaining depending on where you want the raw_data folder
BASE_OUTPUT_DIR = Path.cwd() / "raw_data" / "CMIP6_V1"

def download_variable(client: cdsapi.Client, model: str, variable: str, output_nc: Path) -> None:
    request = {
        "temporal_resolution": "monthly",
        "experiment": "historical",
        "variable": variable,
        "model": model,
        "year": YEARS,
        "month": MONTHS,
        "format": "zip" # FIXED: Explicitly forcing the API to return a zip archive
    }

    with tempfile.TemporaryDirectory() as tmp_dir:
        zip_path = os.path.join(tmp_dir, "download.zip")

        print(f"  Downloading {variable} ...")
        # Added a timeout buffer just in case the server hangs on a single file
        client.retrieve(DATASET, request).download(zip_path)

        print(f"  Extracting .nc file ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            nc_files = [name for name in zf.namelist() if name.endswith(".nc")]
            if not nc_files:
                raise FileNotFoundError(f"No .nc file found in the downloaded zip for {variable}")
            extracted = zf.extract(nc_files[0], tmp_dir)

        shutil.move(extracted, output_nc)
        print(f"  Saved → {output_nc.name}")

def main() -> None:
    client = cdsapi.Client()

    for model in MODELS:
        model_dir = BASE_OUTPUT_DIR / model
        model_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n=== Model: {model} ===")
        for variable in VARIABLES:
            output_nc = model_dir / f"{variable}.nc"

            if output_nc.exists():
                print(f"  Skipping {variable} — already downloaded")
                continue

            try:
                download_variable(client, model, variable, output_nc)
            except Exception as e:
                print(f"  ERROR downloading {variable} for {model}: {e}")

if __name__ == "__main__":
    main()

In [ ]:
import cdsapi

dataset = "projections-cmip6"
request = {
    "temporal_resolution": "monthly",
    "experiment": "ssp2_4_5",
    "variable": [
        "sea_surface_temperature",
        "sea_ice_area_percentage_on_ocean_grid",
        "sea_ice_mass_per_area",
        "sea_ice_thickness",
        "sea_surface_salinity",
        "surface_air_pressure",
        "sea_level_pressure"
    ],
    "model": "mpi_esm1_2_lr",
    "month": [
        "01", "02", "03", "04", "05", "06",
        "07", "08", "09", "10", "11", "12"
    ],
    "year": [
        "2015", "2016", "2017", "2018", "2019", "2020",
        "2021", "2022", "2023", "2024", "2025", "2026",
        "2027", "2028", "2029", "2030", "2031", "2032",
        "2033", "2034", "2035", "2036", "2037", "2038",
        "2039", "2040", "2041", "2042", "2043", "2044",
        "2045", "2046", "2047", "2048", "2049"
    ]
}

print("Initiating connection to Copernicus Climate Data Store...")
client = cdsapi.Client()

print("Request submitted! Waiting in the European server queue...")
# This will save the file directly to your KrillCast_Demo folder
client.retrieve(dataset, request, "mpi_esm1_2_ssp245_future.nc")

print("Download complete! Your future climate data is ready.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

path = Path("/content/drive/MyDrive/KrillCast Cleaned Data/CMIP6_V1/mpi_esm1_2_lr_regridded.nc")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# Load your new, regridded NetCDF file
# ds = xr.open_dataset('your_cleaned_data.nc')
ds = xr.open_dataset(path)


# Select the variable (e.g., 'SST') and a specific time slice (e.g., the first month)
# xarray automatically maps the 2D lat/lon grid for you
ds['SST'].isel(Date=0).plot()
plt.show()

Merging Datasets

In [ ]:
import pandas as pd
import xarray as xr

# 1. Load your two datasets
krill_df = pd.read_csv("/content/drive/MyDrive/KrillCast Cleaned Data/KRILLBASE Data Preprocessing/krillbase_cleaned.csv")
climate_ds = xr.open_dataset("/content/drive/MyDrive/KrillCast Cleaned Data/CMIP6_V1/mpi_esm1_2_lr_regridded.nc")

# Ensure your CSV date column is a proper datetime object so xarray can match it
krill_df['DATE'] = pd.to_datetime(krill_df['DATE'])

# 2. Convert your KRILLBASE coordinates into xarray DataArrays
# This is the magic step that allows xarray to search everything at once
target_dates = xr.DataArray(krill_df['DATE'], dims="points")
target_lats = xr.DataArray(krill_df['LATITUDE'], dims="points")
target_lons = xr.DataArray(krill_df['LONGITUDE'], dims="points")

# 3. Execute the Spatio-Temporal Join
# method='nearest' snaps the krill catch to the closest available climate grid point
matched_climate = climate_ds.sel(
    Date=target_dates,
    lat=target_lats,
    lon=target_lons,
    method='nearest'
)

# 4. Inject the weather data directly back into your KRILLBASE spreadsheet
krill_df['SST'] = matched_climate['SST'].values
# Add any other variables you cleaned (e.g., Sea Ice, Salinity)
# krill_df['SIT'] = matched_climate['SIT'].values

# 5. Check your final matrix!
print(krill_df)

Convert DATE to Fractional_Year

In [ ]:
import pandas as pd

input_file  = "/content/drive/MyDrive/KrillCast Cleaned Data/KRILLBASE Data Preprocessing/krillbase_cleaned.csv"
output_file = "krillbase_cleaned.csv"
cols = ["DATE", "LATITUDE", "LONGITUDE", "NUMBER_OF_KRILL_UNDER_1M2"]

df = pd.read_csv(input_file, encoding='latin-1')

df = df[cols].replace("", pd.NA).dropna(subset=cols)

df["DATE"] = pd.to_datetime(df["DATE"], dayfirst=False, errors="coerce")

df = df.dropna(subset=["DATE"])
df = df.sort_values("DATE")

df.to_csv(output_file, index=False)

output_file = "krillbase_ready_for_merge.csv"

fractional_math = df['DATE'].dt.year + ((df['DATE'].dt.dayofyear - 1) / 365.25)
df.insert(1, "FRACTIONAL_YEAR", fractional_math)

df.to_csv(output_file, index=False)

output_file = "krillbase_cleaned_fractional_year.csv"

df["DATE"] = df['DATE'].dt.year + ((df['DATE'].dt.dayofyear - 1) / 365.25)
df = df.rename(columns={"DATE": "FRACTIONAL_YEAR"})

df.to_csv(output_file, index=False)

In [ ]:
import pandas as pd
import xarray as xr

print("1. Loading datasets...")
# Load the cleaned KRILLBASE data
df = pd.read_csv("krillbase_ready_for_merge.csv")

# Ensure DATE is recognized as a datetime object
df['DATE'] = pd.to_datetime(df['DATE'])

# Load the massive CMIP6 dataset (xarray keeps it on disk, not in RAM)
# Replace 'cmip6_climate_v1.nc' with your actual filename
ds = xr.open_dataset(path)

print("2. Prepping coordinate arrays...")
# Convert pandas columns into xarray DataArrays.
# This is the "magic" step that enables the vectorized Big Data lookup.
time_coords = xr.DataArray(df['DATE'], dims="catch")
lat_coords = xr.DataArray(df['LATITUDE'], dims="catch")
lon_coords = xr.DataArray(df['LONGITUDE'], dims="catch")

print("3. Executing the Spatio-Temporal Join...")
# Extract all 12,000+ weather matches simultaneously
# Make sure the dimension names (time, lat, lon) match exactly what is inside your .nc file!
matched_data = ds.sel(
    Date=time_coords,
    lat=lat_coords,
    lon=lon_coords,
    method="nearest"
)

print("4. Injecting climate data into the KRILLBASE dataframe...")
# Extract the underlying numpy arrays and assign them to new columns
# Update these variable names to match exactly what they are called in your CMIP6 file
df['SST'] = matched_data['SST'].values
df['SIA'] = matched_data['SIA'].values
df['SIM'] = matched_data['SIM'].values  # Swapped LIA for SIM
df['SIT'] = matched_data['SIT'].values
df['SSS'] = matched_data['SSS'].values
df['SAP'] = matched_data['SAP'].values
df['SLP'] = matched_data['SLP'].values

print("5. Saving the final merged dataset...")
df.to_csv("krillcast_merged.csv", index=False)
print("Success: krillcast_merged.csv is ready for tensor conversion!")

# Close the NetCDF file to free up system resources
ds.close()